# Log & Power Transforms (Box-Cox)

_Feature Engineering_

**Squash a long-tailed positive feature so it looks more like a bell curve.**

A heavy-tailed positive feature is lopsided: a tall pile of small values and a long thin tail of giants.

       The logarithm is a ruler that gets coarser as numbers grow. Going from 1 to 10 and from 1000 to 10000 are the same step in log-land (both multiply by 10).

---

This notebook is a practice scaffold for the **Log & Power Transforms (Box-Cox)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## See it for yourself: the problem, then the fix

Run this cell. It plots the **raw** data and reproduces the problem it causes, then plots the **engineered** data and shows the fix — with before/after numbers.

In [ ]:
# =============================================================================
# LESSON: Log / power (Box-Cox) transform of a heavy-tailed positive feature
# -----------------------------------------------------------------------------
# THE STORY a beginner should SEE:
#   - We have ONE positive feature x that is RIGHT-SKEWED: most values are small,
#     but a few are HUGE (a "heavy tail"). Think income, city population, # of
#     followers, web-page visits -- they all look like this.
#   - The TRUE relationship is log-linear:  y grows with log(x), not with x.
#   - If we fit a plain LINEAR REGRESSION on the RAW x, the few giant x values
#     dominate the fit, R^2 is poor, and the residual-vs-fitted plot FUNNELS
#     (heteroscedastic: errors get bigger as the prediction gets bigger).
#   - THE FIX: replace x with log1p(x) = log(1 + x). The feature becomes roughly
#     symmetric/Gaussian, the SAME linear regression now fits well (higher R^2),
#     and the residual plot is an even, flat band.
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# -----------------------------------------------------------------------------
# 1. LOAD / BUILD DATA  (constructed with a FIXED rng so numbers reproduce).
#    We build a REALISTIC right-skewed positive feature: a log-normal variable.
#    log-normal = exp(normal), which is exactly the heavy-tailed shape you get
#    for income / counts / sizes in the real world. The target depends on
#    log(x) -- a genuine log-linear relation -- plus a little noise.
# -----------------------------------------------------------------------------
rng = np.random.default_rng(0)
n = 400

# x is positive and heavy-tailed: most values small, a few enormous.
x = np.exp(rng.normal(loc=0.0, scale=1.3, size=n))      # log-normal -> right-skewed
# TRUE signal is log-linear: y = a + b*log(x) + noise.
y = 3.0 + 2.0 * np.log(x) + rng.normal(scale=1.0, size=n)

X_raw = x.reshape(-1, 1)                                  # sklearn wants 2-D
X_log = np.log1p(x).reshape(-1, 1)                        # log1p = log(1 + x)

# -----------------------------------------------------------------------------
# 2. VISUALIZE THE PURE (RAW) DATA -- histogram of the raw vs the logged feature.
#    The raw histogram is a tall spike near 0 with a long tail to the right.
#    The logged histogram is a nice symmetric bell. (Side by side.)
# -----------------------------------------------------------------------------
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(x, bins=40, color="#d9534f", edgecolor="white")
ax[0].set_title("RAW feature x (right-skewed, heavy tail)")
ax[0].set_xlabel("x"); ax[0].set_ylabel("count")
ax[1].hist(np.log1p(x), bins=40, color="#5cb85c", edgecolor="white")
ax[1].set_title("log1p(x)  (symmetric / ~Gaussian)")
ax[1].set_xlabel("log1p(x)"); ax[1].set_ylabel("count")
plt.tight_layout(); plt.show()

# -----------------------------------------------------------------------------
# 3. REPRODUCE THE PROBLEM ON RAW DATA -- fit linear regression on RAW x.
#    R^2 is poor because a straight line in x cannot follow a log-shaped curve,
#    and the handful of giant x values drag the line around.
# -----------------------------------------------------------------------------
raw_model = LinearRegression().fit(X_raw, y)
y_pred_raw = raw_model.predict(X_raw)
r2_raw = r2_score(y, y_pred_raw)
resid_raw = y - y_pred_raw
print(f"PROBLEM (raw x):        R^2 = {r2_raw:.3f}")

# -----------------------------------------------------------------------------
# 4. APPLY THE FIX (already computed X_log above) and refit the SAME model.
# -----------------------------------------------------------------------------
log_model = LinearRegression().fit(X_log, y)
y_pred_log = log_model.predict(X_log)
r2_log = r2_score(y, y_pred_log)
resid_log = y - y_pred_log
print(f"FIX (log1p(x) feature): R^2 = {r2_log:.3f}")

# -----------------------------------------------------------------------------
#    VISUALIZE THE FAILURE vs THE FIX -- residual-vs-fitted scatter, side by side.
#    LEFT (raw): a FUNNEL -- spread of residuals grows with the fitted value
#                (heteroscedastic). The model is systematically wrong.
#    RIGHT (log): an EVEN, flat band of points around 0 -- well-behaved errors.
# -----------------------------------------------------------------------------
fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
ax[0].scatter(y_pred_raw, resid_raw, s=12, alpha=0.6, color="#d9534f")
ax[0].axhline(0, color="black", lw=1)
ax[0].set_title(f"RAW: residuals FUNNEL  (R^2={r2_raw:.2f})")
ax[0].set_xlabel("fitted value"); ax[0].set_ylabel("residual (y - prediction)")
ax[1].scatter(y_pred_log, resid_log, s=12, alpha=0.6, color="#5cb85c")
ax[1].axhline(0, color="black", lw=1)
ax[1].set_title(f"FIXED: residuals EVEN  (R^2={r2_log:.2f})")
ax[1].set_xlabel("fitted value")
plt.tight_layout(); plt.show()

# -----------------------------------------------------------------------------
# 5. ONE-LINE SUMMARY
# -----------------------------------------------------------------------------
print(f"PROBLEM (raw): {r2_raw:.3f}   ->   FIX (log1p feature): {r2_log:.3f}")

## Reference implementation — pandas / numpy / scikit-learn / scipy

In [ ]:
import pandas as pd
import numpy as np
import json
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

# ---------------------------------------------------------------
# Example 1: Yelp business review counts (Yelp Dataset Challenge)
#   data: github.com/alicezheng/feature-engineering-book
# ---------------------------------------------------------------
biz_file = open('yelp_academic_dataset_business.json')
biz_df = pd.DataFrame([json.loads(x) for x in biz_file.readlines()])
biz_file.close()

# Log transform: add 1 first so review_count == 0 -> log10(1) == 0
biz_df['log_review_count'] = np.log10(biz_df['review_count'] + 1)

# Does the log feature predict star rating better than the raw count?
# 10-fold cross-validated R^2 of a one-feature linear regression.
m_orig = LinearRegression()
scores_orig = cross_val_score(m_orig, biz_df[['review_count']],
                              biz_df['stars'], cv=10)

m_log = LinearRegression()
scores_log = cross_val_score(m_log, biz_df[['log_review_count']],
                             biz_df['stars'], cv=10)

print("R^2 raw review_count : %0.5f (+/- %0.5f)"
      % (scores_orig.mean(), scores_orig.std() * 2))
print("R^2 log_review_count : %0.5f (+/- %0.5f)"
      % (scores_log.mean(), scores_log.std() * 2))
# Book result: both about -0.037 -- the histogram is reshaped a lot,
# but this weak single-feature model barely changes.

# ---------------------------------------------------------------
# Example 2: UCI Online News Popularity word-count feature
#   archive.ics.uci.edu/ml/datasets/Online+News+Popularity
# ---------------------------------------------------------------
news_df = pd.read_csv('OnlineNewsPopularity.csv', delimiter=', ')

news_df['log_n_tokens_content'] = np.log10(news_df['n_tokens_content'] + 1)

m_orig = LinearRegression()
scores_orig = cross_val_score(m_orig, news_df[['n_tokens_content']],
                              news_df['shares'], cv=10)
m_log = LinearRegression()
scores_log = cross_val_score(m_log, news_df[['log_n_tokens_content']],
                             news_df['shares'], cv=10)

print("R^2 raw n_tokens_content : %0.5f" % scores_orig.mean())
print("R^2 log n_tokens_content : %0.5f" % scores_log.mean())
# Book result: improves from -0.00242 (raw) to -0.00114 (logged):
# the log made the word-count feature much more Gaussian and slightly
# improved the linear regression's R^2.

# ---------------------------------------------------------------
# Example 3: Box-Cox on the Yelp review counts (needs positive data,
#   so feed review_count + 1). boxcox picks lambda by max likelihood.
# ---------------------------------------------------------------
rc_log = np.log10(biz_df['review_count'] + 1)
rc_bc, bc_params = stats.boxcox(biz_df['review_count'] + 1)
print("Box-Cox lambda = %0.4f" % bc_params)   # ~0 means ~ a log transform

## Visualize the data & results

_Take a heavy-tailed positive feature (the 'mean area' of breast-cancer tumors). Does a log transform turn its lopsided distribution into something more symmetric and bell-shaped?_

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer

# A heavy-tailed positive feature: tumor 'mean area' (column 3).
bc = load_breast_cancer()
area = bc.data[:, 3]                      # strictly positive, right-skewed

# Subsample to 60 points for a clean in-browser histogram.
rng = np.random.RandomState(0)
area = rng.choice(area, size=60, replace=False)

# BEFORE: histogram of the raw feature (7 equal-width bins).
raw_counts, raw_edges = np.histogram(area, bins=7)

# AFTER: histogram of log10(area). All values > 0, so no +1 needed here;
# for count features with zeros you would use np.log10(x + 1).
log_area = np.log10(area)
log_counts, log_edges = np.histogram(log_area, bins=7)

print("raw skew (right tail):", round(float(((area - area.mean())**3).mean()
      / area.std()**3), 3))
print("log skew (more symmetric):", round(float(((log_area - log_area.mean())**3).mean()
      / log_area.std()**3), 3))
print("raw  bin counts:", raw_counts.tolist())
print("log  bin counts:", log_counts.tolist())
# Raw distribution is right-skewed; log10 pulls the tail in and the
# histogram becomes far more symmetric / Gaussian -- exactly the book's point.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A feature page_views is a count that ranges from 0 to about 4,000,000, with most pages under 100 views. You want to feed it to a linear regression. Your colleague writes np.log10(df['page_views']) and the pipeline crashes on some rows. What broke, what is the one-character-idea fix, and why does logging help this feature at all?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Find what crashed. — _Some pages have 0 views, and $\log_{10}(0)=-\infty$ (NumPy emits $-\infty$ / a warning, and downstream math on $-\infty$ blows up). Any zero-containing count feature hits this._
- Apply the book's fix. — _Add 1 before logging: np.log10(df['page_views'] + 1) (or np.log1p). Now a count of 0 maps to $\log_{10}(1)=0$, finite and sensible. The "+1" is the whole trick._
- Explain why logging is worth it here. — _Page views span six orders of magnitude with a giant tail. The log compresses the millions-of-views giants and spreads the under-100 crowd, turning a spike-plus-tail into a more symmetric, more Gaussian feature that a linear model handles better._

**Answer:** The crash is $\log_{10}(0)=-\infty$ — the count feature has zeros. The fix is the book's $\log_{10}(x+1)$: np.log10(df['page_views'] + 1), which sends 0 to 0 instead of $-\infty$. Logging helps because page_views is heavy-tailed over many orders of magnitude; the log squashes the giants and stretches the small bulk, producing a more symmetric, more Gaussian feature that linear regression prefers.

</details>

**Problem 2.** You run scipy.stats.boxcox on a strictly-positive feature and it returns $\lambda \approx 0.02$. (a) What transform is Box-Cox essentially applying? (b) What would $\lambda \approx 0.5$ have meant instead? (c) A teammate fits $\lambda$ on the full dataset (train + test together). Why is that a mistake?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Read $\lambda \approx 0$. — _Box-Cox at $\lambda=0$ is exactly $\ln(x)$. A fitted $\lambda\approx 0.02$ means maximum likelihood decided the feature is heavy-tailed enough that an essentially-log transform makes it most Gaussian._
- Contrast $\lambda \approx 0.5$. — _$\lambda=0.5$ is a (shifted, scaled) square root — a gentler squash, the natural variance-stabilizer for milder, Poisson-like count data._
- Spot the leak. — _$\lambda$ is a parameter estimated from data. Fitting it on test rows lets test information bleed into the transform, so your reported test score is optimistic. Fit $\lambda$ on train only, then apply that fixed $\lambda$ to validation/test — like any scaler._

**Answer:** (a) $\lambda\approx 0$ is the log: Box-Cox at $\lambda=0$ is $\ln(x)$, so it's applying essentially a natural-log transform. (b) $\lambda\approx 0.5$ would be a square-root-like squash — gentler, the variance-stabilizing choice for milder Poisson-like counts. (c) Fitting $\lambda$ on train+test leaks test information into a learned parameter, inflating the test score. Estimate $\lambda$ on the training set only and reuse that same $\lambda$ on validation and test.

</details>